# misc

> Sundry utilities that don't fit a bigger category yet: reading PDFs and checking local GPU availability. (Web fetching, GitHub CI checks, and remote GPU orchestration live in `remote` -- they're all about reaching outside the local machine.)

In [ ]:
#| default_exp misc

In [ ]:
#| export
import subprocess, os
import pymupdf
from slmn.nbtools import _resolve_safe

## read_pdf

Extract plain text from a PDF over a 1-indexed page range (via PyMuPDF) -- e.g. to let an
LLM read a paper. Paths go through `nbtools._resolve_safe`, so the sandbox setting applies
here too.

In [ ]:
#| export
def read_pdf(path:str, # path to the PDF file
             start_page:int, # first page to extract, 1-indexed
             end_page:int # last page to extract, inclusive
             ) -> str:
    "Extract plain text from a PDF file, by 1-indexed page range."
    path = _resolve_safe(path)
    doc = pymupdf.open(path)
    out = []
    for i in range(start_page - 1, min(end_page, len(doc))):
        out.append(f"\n--- Page {i+1} ---\n{doc[i].get_text()}")
    return ''.join(out)

## gpu_free

Is a local GPU actually free for compute? Tries `nvidia-smi`, falls back to checking
`/dev/nvidia*` with `fuser` (works on some WSL setups where `nvidia-smi` is absent), and
if there's no GPU tooling at all, assumes free. Returns `{'free': bool, 'detail': str}`.

In [ ]:
#| export
def gpu_free(gpu_idx:int=0, # which GPU to check
             busy_threshold_mib:int=2000 # VRAM usage (MiB) below which the GPU still counts as free
             ) -> dict:
    "Check whether a local GPU is free for compute. Tries nvidia-smi first; falls back to checking /dev/nvidia* via fuser (works on some WSL setups); if neither is available, assumes free. Returns {'free': bool, 'detail': str}."
    nvidia_smi = subprocess.run(['which', 'nvidia-smi'], capture_output=True, text=True).stdout.strip()
    if nvidia_smi:
        r = subprocess.run(['nvidia-smi', '-i', str(gpu_idx), '--query-compute-apps=pid,used_gpu_memory,name',
                             '--format=csv,noheader'], capture_output=True, text=True)
        apps = r.stdout.strip()
        if not apps:
            return {'free': True, 'detail': f'GPU {gpu_idx}: FREE'}
        total_mib = 0
        procs = []
        for line in apps.splitlines():
            pid, mem, name = [x.strip() for x in line.split(',', 2)]
            mib = int(''.join(c for c in mem if c.isdigit()) or 0)
            total_mib += mib
            procs.append(f"PID {pid} | {mem} | {name}")
        if total_mib < busy_threshold_mib:
            return {'free': True, 'detail': f'GPU {gpu_idx}: FREE ({total_mib} MiB in use, below {busy_threshold_mib} MiB threshold)'}
        return {'free': False, 'detail': f'GPU {gpu_idx}: BUSY ({total_mib} MiB in use)\n  ' + '\n  '.join(procs)}

    if os.path.exists('/dev') and any(f.startswith('nvidia') for f in os.listdir('/dev')):
        r = subprocess.run(['fuser'] + [f'/dev/{f}' for f in os.listdir('/dev') if f.startswith('nvidia')],
                            capture_output=True, text=True)
        procs = r.stdout.split()
        if not procs:
            return {'free': True, 'detail': f'GPU {gpu_idx}: FREE (nvidia-smi unavailable; no /dev/nvidia* users)'}
        return {'free': False, 'detail': f'GPU {gpu_idx}: BUSY (nvidia-smi unavailable; /dev/nvidia* in use by PIDs: {" ".join(procs)})'}

    return {'free': True, 'detail': f'GPU {gpu_idx}: FREE (nvidia-smi unavailable; no GPU device files found; assuming free)'}